# Desktop Indicator

In [ ]:
import threading
import time
from pyvda import VirtualDesktop, get_virtual_desktops
from PIL import Image, ImageDraw, ImageFont
import pystray


from PIL import Image, ImageDraw, ImageFont

def make_icon(number: int) -> Image.Image:
    """Return a 32×32 tray-icon with the desktop number centered in it."""
    size = (32, 32)
    img = Image.new("RGBA", size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(img)

    # 1) background circle
    draw.ellipse((0, 0, size[0], size[1]), fill=(0, 120, 215, 255))

    # 2) prepare text
    font = ImageFont.load_default()
    text = str(number)

    # 3) measure text with textbbox
    #    bbox = (left, top, right, bottom)
    bbox = draw.textbbox((0, 0), text, font=font)
    w = bbox[2] - bbox[0]
    h = bbox[3] - bbox[1]

    # 4) compute position to center
    x = (size[0] - w) // 2
    y = (size[1] - h) // 2

    # 5) draw the number
    draw.text((x, y), text, font=font, fill="white")

    return img



def monitor(icon: pystray.Icon):
    """Thread-loop: check every second, update icon if desktop changed."""
    last = None
    while True:
        try:
            current = VirtualDesktop.current().number
        except Exception:
            current = None
        if current != last:
            icon.icon = make_icon(current)
            icon.title = f"Desktop #{current}"
            last = current
        time.sleep(1)

def main():
    # create tray-icon
    icon = pystray.Icon("vd_indicator")
    # initialize with current desktop
    num = VirtualDesktop.current().number
    icon.icon = make_icon(num)
    icon.title = f"Desktop #{num}"
    # start monitor thread
    t = threading.Thread(target=monitor, args=(icon,), daemon=True)
    t.start()
    # run the tray icon loop
    icon.run()

if __name__ == "__main__":
    main()
